# Additional BAP scenario — OR-Tools CP-SAT

Install `scenarios/bap_cpsat/requirements.txt` in the active kernel, then run all
cells. This notebook reuses the published CP-SAT adapter and existing reference
code. It does not edit the original classical/quantum notebooks or reports.


In [ ]:
from pathlib import Path
import sys
import json
import runpy

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/hypaq_ports/classical/cpsat.py").is_file()
             and (p / "scenarios/bap_cpsat/compare.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside hypaq-ports after adding scenarios/bap_cpsat.")
SCENARIO = ROOT / "scenarios/bap_cpsat"
sys.path.insert(0, str(ROOT / "src"))
from hypaq_ports.reference import load_json
comparison = runpy.run_path(str(SCENARIO / "compare.py"))
print("Repository:", ROOT)


## Compare against enumeration on the same inputs

The original instance has zero optimal waiting. The additional instance has an
optimum of one waiting slot. CP-SAT and enumeration may select different tied
schedules. Feasibility and objective are compared independently of tie-breaking.


In [ ]:
for path, expected in (
    (ROOT / "data/synthetic/berth_tiny_v1.json", 0),
    (SCENARIO / "data/berth_waiting_v1.json", 1),
):
    report = comparison["generate_comparison"](load_json(path))
    assert report["reference_optimal_waiting_slots"] == expected
    assert all(row["status"] == "OPTIMAL" for row in report["summary"])
    assert all(row["matches_reference_optimum"] for row in report["summary"])
    print(report["instance_id"])
    print(json.dumps(report["summary"], indent=2))
    print("CP-SAT schedule:", json.dumps(report["results"]["cpsat"]["schedule"], indent=2))


## Optional quantum comparison

The additional `compare.py --include-quantum` command runs the unchanged QAOA and
FALQON implementations using the existing Ket/KBW configuration. Install the
repository's `requirements-quantum.txt` first; see this folder's README for exact
commands. Quantum rows remain HEURISTIC even when a sample matches the known
optimum. The notebook itself generates no files. CPU simulation, differing
budgets and these tiny inputs do not establish quantum advantage.
